# Declarações de inidoneidade nos acórdãos

Extrai, de `vw_ia_votos_acordaos_decisoes`, os acórdãos que **declaram a inidoneidade** de licitante
(LCE 464/2012, art. 109; RI, art. 325 — até 5 anos) com LLM via LangChain e schema Pydantic.

LGPD: o modelo é o deployment `gpt-5.2` do Foundry do SERPRO, sempre via `ccd.llm`.

In [ ]:
import re
import unicodedata
from typing import Literal

import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from ccd.config import OUTPUT_DIR
from ccd.db import run_query_df
from ccd.llm import get_llm, structured
from ccd.notebook import setup

ctx = setup(llm=False)
MODELO = "gpt-5.2"  # deployment no Foundry do SERPRO (implantados: gpt-5-mini, gpt-5.1, gpt-5.2; "gpt-5" não)
llm = get_llm(model=MODELO, temperature=None)  # gpt-5.x só aceita a temperatura padrão

## Schema

In [ ]:
class DeclaracaoInidoneidade(BaseModel):
    """Uma pessoa declarada inidônea no dispositivo do acórdão."""

    nome: str = Field(description="Nome da pessoa física ou razão social da pessoa jurídica declarada inidônea.")
    cpf_cnpj: str | None = Field(None, description="CPF ou CNPJ exatamente como consta no texto; null se não constar.")
    tipo_pessoa: Literal["física", "jurídica"] | None = Field(None, description="Tipo de pessoa; null se não for possível inferir.")
    prazo_anos: int | None = Field(None, description="Prazo da inidoneidade em anos inteiros; null se não constar ou não for inteiro (detalhar em prazo_texto).")
    prazo_texto: str | None = Field(None, description="Prazo como escrito no texto (ex.: 'pelo prazo de 5 (cinco) anos').")
    abrangencia: str | None = Field(None, description="Alcance da sanção (ex.: 'licitações na administração pública estadual e municipal').")
    motivo: str | None = Field(None, description="Conduta que motivou a sanção (ex.: 'fraude à licitação', 'conluio entre licitantes').")
    fundamento_legal: str | None = Field(None, description="Dispositivos legais citados para a sanção (ex.: 'art. 109 da LCE 464/2012').")
    trecho: str = Field(description="Trecho do dispositivo que declara a inidoneidade, copiado LITERALMENTE do texto, sem resumir.")


class ExtracaoInidoneidade(BaseModel):
    """Resultado da leitura de um acórdão."""

    declara_inidoneidade: bool = Field(description="True somente se o dispositivo do acórdão efetivamente declara a inidoneidade de alguém.")
    declaracoes: list[DeclaracaoInidoneidade] = Field(default_factory=list, description="Uma entrada por pessoa declarada inidônea; vazia se declara_inidoneidade for False.")
    observacao: str | None = Field(None, description="Breve justificativa quando não há declaração, ou ressalva relevante (ex.: inidoneidade afastada, apenas proposta).")

## Prompt

In [ ]:
SISTEMA = """Você é analista da Coordenadoria de Controle de Decisões (CCD) do Tribunal de Contas do Estado do Rio Grande do Norte.
Sua tarefa é identificar, no acórdão, a DECLARAÇÃO DE INIDONEIDADE de licitante para participar de licitação / contratar com a administração pública (LCE 464/2012, art. 109; Regimento Interno, art. 325).

Regras:
1. Considere apenas o que o Tribunal efetivamente DECIDIU: o dispositivo do acórdão (texto do acórdão) e, se ele remeter ao voto, a conclusão do voto vencedor.
2. NÃO são declaração de inidoneidade:
   - inabilitação para cargo em comissão ou função de confiança (art. 1º, XXV, "a" do RI) — é outra sanção;
   - menções no relatório, na fundamentação, em pareceres do Corpo Técnico ou do Ministério Público de Contas que não foram acolhidas no dispositivo;
   - "deixo de declarar", afastamento ou exclusão da sanção, recomendação de apuração ou de instauração de processo;
   - representação/comunicação a outro órgão para que ELE aplique sanção.
3. Gere uma entrada por pessoa (física ou jurídica) declarada inidônea.
4. Copie o campo "trecho" literalmente do texto fornecido. Não invente CPF, CNPJ, prazo ou fundamento: use null quando não constar.
5. Se não houver declaração, retorne declara_inidoneidade=false, declaracoes vazia e explique em observacao."""

USUARIO = """Processo: {processo}
Acórdão/Decisão: {acordao}

EMENTA:
{ementa}

CONCLUSÃO DO VOTO:
{conclusao}

TEXTO DO ACÓRDÃO:
{texto_acordao}"""

prompt = ChatPromptTemplate.from_messages([("system", SISTEMA), ("human", USUARIO)])
chain = prompt | structured(ExtracaoInidoneidade, llm=llm)

## Acórdãos candidatos

Pré-filtro por `LIKE` (barato) só no voto escolhido; o LLM separa declaração de mera menção.

In [ ]:
SQL = """
SELECT idvoto, IdProcesso, NumeroProcesso, AnoProcesso, numeroResultado, anoResultado, resultadoTipo,
       decisao, CodigoCamara, DataSessao, DataEncerramentoSessao, DataDecisaoMonocratica,
       NomeRelator, OrgaoOrigem, interessado, ementa, Conclusao, texto_acordao
FROM vw_ia_votos_acordaos_decisoes
WHERE VotoEscolhido = 1
  AND (texto_acordao LIKE :padrao OR Conclusao LIKE :padrao OR ementa LIKE :padrao)
"""
df = run_query_df(SQL, padrao="%inid%ne%")
df = df.apply(lambda c: c.str.strip() if c.dtype == object else c)
print(len(df), "votos;", df.IdProcesso.nunique(), "processos")
df.head()

## Extração

In [ ]:
entradas = [
    {
        "processo": f"{r.NumeroProcesso}/{r.AnoProcesso}",
        "acordao": f"{r.numeroResultado}/{r.anoResultado} ({r.resultadoTipo})",
        "ementa": r.ementa or "",
        "conclusao": r.Conclusao or "",
        "texto_acordao": r.texto_acordao or "",
    }
    for r in df.itertuples()
]
resultados = chain.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)
erros = sum(isinstance(r, Exception) for r in resultados)
print(f"{len(resultados) - erros} ok, {erros} erros")

## Conferência do trecho

O LLM pode "citar" o que não está no texto: marca se o trecho existe na fonte (normalizado).

In [ ]:
def normaliza(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode()
    return re.sub(r"\W+", " ", s.casefold()).strip()


assert normaliza("Declarar  a INIDONEIDADE,\n por 5 anos") == normaliza("declarar a inidoneidade por 5 anos")

linhas, negativos = [], []
for r, res in zip(df.itertuples(), resultados):
    base = {
        "idvoto": r.idvoto,
        "processo": f"{r.NumeroProcesso}/{r.AnoProcesso}",
        "acordao": f"{r.numeroResultado}/{r.anoResultado}",
        "data": r.DataSessao if pd.notna(r.DataSessao) else r.DataDecisaoMonocratica,
        "relator": r.NomeRelator,
        "orgao": r.OrgaoOrigem,
    }
    if isinstance(res, Exception):
        negativos.append({**base, "erro": repr(res)})
        continue
    if not res.declara_inidoneidade or not res.declaracoes:
        negativos.append({**base, "observacao": res.observacao})
        continue
    fonte = normaliza(" ".join(filter(None, [r.ementa, r.Conclusao, r.texto_acordao])))
    for d in res.declaracoes:
        linhas.append({**base, **d.model_dump(), "trecho_conferido": normaliza(d.trecho) in fonte})

inidoneos = pd.DataFrame(linhas)
revisar = pd.DataFrame(negativos)
print(len(inidoneos), "declarações;", (~inidoneos.get("trecho_conferido", pd.Series(dtype=bool))).sum(), "com trecho não conferido")
inidoneos

In [ ]:
revisar  # sem declaração ou com erro — conferir alguns à mão

## Datas e origem da informação

- **Evento**: `SequencialProcessoEvento` da informação do acórdão (ordem = sufixo `_NNNN` do arquivo da decisão).
- **Publicação**: `DataPublicacaoDOE` da sessão em que o acórdão foi julgado.
- **Trânsito em julgado**: `Processo_TransitoJulgado` (ativo) da decisão.
- **Início/fim da sanção**: início = trânsito em julgado; fim = início + prazo. Sem trânsito, ficam em branco.

In [ ]:
SQL_EVENTO = """
SELECT TOP 1 e.SequencialProcessoEvento
FROM vw_ata_informacao a
JOIN Pro_ProcessoEvento e ON e.IdInformacao = a.idInformacao
WHERE a.numero_processo = :n AND a.ano_processo = :a AND a.ordem = :ordem
"""
SQL_DOE = """
SELECT MIN(DataPublicacaoDOE) FROM vw_DOEPublicacaoEncerramentoSessao
WHERE numero_processo = :n AND ano_processo = :a AND codigo_camara = :camara
  AND CAST(data_encerramento_sessao AS date) = CAST(:encerramento AS date)
"""
SQL_TRANSITO = """
SELECT MAX(datatransito) FROM Processo_TransitoJulgado
WHERE numero_processo = :n AND ano_processo = :a AND inativo = 0
  AND TRY_CAST(numerodecisao AS int) = :num AND anodecisao = :ano
"""
TIPO = {"A": "Acórdão", "D": "Decisão"}


def escalar(sql: str, **params):
    v = run_query_df(sql, **params).iloc[0, 0]
    return None if pd.isna(v) else v


def datas_acordao(r) -> dict:
    p = {"n": r.NumeroProcesso, "a": r.AnoProcesso}
    ordem = re.search(r"_(\d+)\.pdf$", r.decisao or "")
    evento = escalar(SQL_EVENTO, **p, ordem=int(ordem.group(1))) if ordem else None
    publicacao = (
        escalar(SQL_DOE, **p, camara=r.CodigoCamara, encerramento=r.DataEncerramentoSessao)
        if pd.notna(r.DataEncerramentoSessao) else None
    )
    transito = escalar(SQL_TRANSITO, **p, num=int(r.numeroResultado), ano=r.anoResultado)
    tipo = TIPO.get(r.resultadoTipo, "Decisão")
    origem = (
        f"{tipo} nº {int(r.numeroResultado)}/{r.anoResultado}-TC, Processo TCE/RN nº "
        f"{r.NumeroProcesso}/{r.AnoProcesso}" + (f", Evento {int(evento)}" if evento else "")
    )
    return {"idvoto": r.idvoto, "evento": evento, "data_publicacao": publicacao,
            "data_transito": transito, "origem_informacao": origem}


# ponytail: 3 consultas por acórdão com declaração (poucos); vira JOIN se o volume crescer
datas = pd.DataFrame([datas_acordao(r) for r in df[df.idvoto.isin(inidoneos.get("idvoto", []))].itertuples()])
if len(inidoneos):
    inidoneos = inidoneos.merge(datas, on="idvoto", how="left")
    inidoneos["inicio_sancao"] = pd.to_datetime(inidoneos.data_transito)
    inidoneos["fim_sancao"] = [
        i + pd.DateOffset(years=int(p)) if pd.notna(i) and pd.notna(p) else pd.NaT
        for i, p in zip(inidoneos.inicio_sancao, inidoneos.prazo_anos)
    ]
inidoneos

## CPF/CNPJ pelas partes do processo

Quando o acórdão não traz o documento, procura o declarado entre as pessoas associadas ao processo
(`Pro_ProcessosResponsavelDespesa` → `GenPessoa`), casando pelo nome normalizado (sem sufixos societários).

In [ ]:
SQL_PARTES = """
SELECT DISTINCT p.Nome, p.Documento
FROM Pro_ProcessosResponsavelDespesa r
JOIN GenPessoa p ON p.IdPessoa = r.IdPessoa
WHERE r.Numero_Processo = :n AND r.Ano_Processo = :a AND p.Documento IS NOT NULL
"""
# palavras que não identificam a pessoa: sufixo societário, artigos e preposições
IGNORAR = {"ltda", "me", "epp", "eireli", "sa", "s", "a", "cia", "de", "da", "do", "das", "dos", "e", "em", "o"}


def palavras(s: str) -> set[str]:
    s = re.split(r"\s[-–]\s|,|\bpor seus?\b", s or "", flags=re.I)[0]  # corta "- Por seu gestor ..."
    s = re.sub(r"\brio grande do norte\b|\br n\b", "rn", normaliza(s))
    return {t for t in s.split() if t not in IGNORAR}


def semelhanca(x: str, y: str) -> float:
    """Sobreposição de palavras (|A∩B| / menor conjunto), tolerante a ordem e siglas extras."""
    a, b = palavras(x), palavras(y)
    return len(a & b) / min(len(a), len(b)) if min(len(a), len(b)) >= 2 else 0.0


def doc_nas_partes(nome: str, partes: pd.DataFrame) -> tuple[str | None, str | None]:
    """(documento, nome cadastrado) da parte mais parecida; None se nenhuma chegar a 0,85."""
    notas = [semelhanca(nome, p) for p in partes.Nome]
    if not notas or max(notas) < 0.85:  # ponytail: limiar fixo; conferir a coluna "Fonte do CPF/CNPJ" se houver homônimos
        return None, None
    melhor = partes.iloc[notas.index(max(notas))]
    return melhor.Documento.strip(), melhor.Nome.strip()


_p = pd.DataFrame({
    "Nome": ["MOVETECH COMÉRCIO E REPRESENTAÇÕES LTDA", "EMMO ENGENHARIA LTDA",
             "Instituto de Formação e Gestão em Turismo do R N - Por seu gestor Robespierre do O Procópio Barreto"],
    "Documento": ["01677278000160", "05577398000147", "06172370000192"],
})
assert doc_nas_partes("MOVETECH Comércio e Representações LTDA.", _p)[0] == "01677278000160"
assert doc_nas_partes("INSTITUTO DE GESTÃO E FORMAÇÃO EM TURISMO DO RIO GRANDE DO NORTE – IGETUR/RN", _p)[0] == "06172370000192"
assert doc_nas_partes("TCL Limpeza Urbana Ltda.", _p) == (None, None)

if len(inidoneos):
    inidoneos["cpf_cnpj"] = inidoneos.cpf_cnpj.str.replace(r"\D", "", regex=True).replace("", None)  # só dígitos
    inidoneos["fonte_cpf_cnpj"] = inidoneos.cpf_cnpj.notna().map({True: "texto do acórdão", False: None})
    for i, r in inidoneos[inidoneos.cpf_cnpj.isna()].iterrows():
        n, a = r.processo.split("/")
        doc, nome_cadastro = doc_nas_partes(r.nome, run_query_df(SQL_PARTES, n=n, a=a))
        if doc:
            inidoneos.loc[i, ["cpf_cnpj", "fonte_cpf_cnpj"]] = [doc, f"partes do processo ({nome_cadastro})"]
inidoneos[["processo", "nome", "cpf_cnpj", "fonte_cpf_cnpj"]]

## Saída

In [ ]:
COLUNAS = {
    "processo": "Processo (nº/ano)",
    "acordao": "Acórdão",
    "nome": "Declarado inidôneo",
    "cpf_cnpj": "CPF/CNPJ",
    "fonte_cpf_cnpj": "Fonte do CPF/CNPJ",
    "tipo_pessoa": "Tipo de pessoa",
    "prazo_anos": "Prazo (anos)",
    "inicio_sancao": "Início da sanção",
    "fim_sancao": "Fim da sanção",
    "data_publicacao": "Publicação da sanção (DOE)",
    "data_transito": "Trânsito em julgado",
    "origem_informacao": "Origem da informação",
    "motivo": "Motivo",
    "abrangencia": "Abrangência",
    "fundamento_legal": "Fundamento legal",
    "trecho": "Trecho do acórdão",
    "trecho_conferido": "Trecho conferido",
    "data": "Data da sessão",
    "relator": "Relator",
    "orgao": "Órgão",
}
tabela = inidoneos.reindex(columns=list(COLUNAS)).rename(columns=COLUNAS)
for c in ["Início da sanção", "Fim da sanção", "Publicação da sanção (DOE)", "Trânsito em julgado", "Data da sessão"]:
    tabela[c] = pd.to_datetime(tabela[c]).dt.date

saida = OUTPUT_DIR / "analise" / "inidoneidade.xlsx"
saida.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(saida) as xw:
    tabela.to_excel(xw, sheet_name="inidoneidade", index=False)
    revisar.to_excel(xw, sheet_name="sem_declaracao", index=False)
saida